# Setup

In [4]:
import time
import pandas as pd
import tqdm
import boto3
import io 

In [2]:
def check_s3_prefix_exists(bucket_name, s3_prefix, source_id, specific_file = "annotations.csv"):
    s3 = boto3.client("s3")
    prefix = f"{s3_prefix}/s{source_id}/{specific_file}"

    response = s3.list_objects_v2(Bucket=bucket_name, Prefix=prefix, MaxKeys=1)

    if "Contents" in response:
        # print(f"Prefix exists: {prefix}")
        return True
    else:
        # print(f"Prefix does not exist: {prefix}")
        return False

# Data

In [3]:
bucket_name = "dev-datamermaid-sm-sources"
prefix = "coralnet-public-images"
s3 = boto3.client("s3")

In [4]:
df_counts = pd.read_csv("dataframes/coralnet_source_counts.csv")
df_counts

,source_id,image_list_count,annotations_count,images_folder_count
0,23,750,750,750
1,57,1681,1681,1681
2,69,100,100,100
3,70,300,300,300
4,82,1,0,1
...,...,...,...,...
1956,8277,0,0,0
1957,8278,0,0,0
1958,8284,0,0,0
1959,8288,0,0,0


# Get a list of all annotations and assign status (Confirmed, Unconfirmed, Unclassified) with an associated image

In [5]:
start_time = time.time()
df_annotation_list = []

for row_i, row in tqdm.tqdm(df_counts.iterrows()):
    # if row_i==1000:
    #     break
    source_id = row["source_id"]
    problematic = row["image_list_count"]<row["images_folder_count"]

    annotations_flag = check_s3_prefix_exists(
        bucket_name=bucket_name, s3_prefix=prefix, source_id=source_id, specific_file="annotations.csv"
    )
    if not annotations_flag:
        # print(f"Source {source_id} does not have an annotations.csv file. Skipping.")
        continue

    image_list_flag = check_s3_prefix_exists(
        bucket_name=bucket_name, s3_prefix=prefix, source_id=source_id, specific_file="image_list.csv"
    )
    if not image_list_flag:
        # print(f"Source {source_id} does not have an image_list.csv file. Skipping.")
        continue


    df_annotations = pd.read_csv(
        f"s3://{bucket_name}/{prefix}/s{source_id}/annotations.csv", low_memory=False
    )
    

    if problematic:
        s3_key_output = f"{prefix}/imagelist/s{source_id}_image_list.csv"
        obj_il = s3.get_object(Bucket=bucket_name, Key=s3_key_output)
        df_images = pd.read_csv(obj_il["Body"])
    else:
        df_images = pd.read_csv(
            f"s3://{bucket_name}/{prefix}/s{source_id}/image_list.csv", low_memory=False  # Perhaps this is unnecessary and can just use tha annotations as in Mermaid
        )

    df_images["Status"] = "Other"
    confirmed_mask = df_images["Name"].apply(lambda x: "Confirmed" in x)
    Unconfirmed_mask = df_images["Name"].apply(lambda x: "Unconfirmed" in x)
    Unclassified_mask = df_images["Name"].apply(lambda x: "Unclassified" in x)
    df_images.loc[confirmed_mask, "Status"] = "Confirmed"
    df_images.loc[Unconfirmed_mask, "Status"] = "Unconfirmed"
    df_images.loc[Unclassified_mask, "Status"] = "Unclassified"

    df_images["Name"] = df_images["Name"].apply(
        lambda x: x.replace(" - Confirmed", "").replace(" - Unconfirmed", "").replace(" - Unclassified", "")
    )
    df_images["image_id"] = df_images["Image Page"].apply(
        lambda x: x.replace("/image/", "").replace("/view/", "")
    )
    df_annotations = pd.merge(
        df_annotations,
        df_images,
        left_on="Name",
        right_on="Name",
        how="inner",
        suffixes=("", "_y"),
    )
    df_annotations["source_id"] = source_id
    df_annotation_list.append(df_annotations[["source_id", "image_id", "Status", "Row", "Column", "Label ID"]])

df_annotations = pd.concat(
    df_annotation_list, ignore_index=True
)

end_time = time.time()
print("Time taken (seconds):", end_time - start_time)

1961it [10:45,  3.04it/s]


Time taken (seconds): 651.9910275936127


In [7]:
df_annotations

,source_id,image_id,Status,Row,Column,Label ID
0,23,12805,Confirmed,735,1008,112
1,23,12805,Confirmed,663,1682,106
2,23,12805,Confirmed,955,1737,106
3,23,12805,Confirmed,1034,1431,105
4,23,12805,Confirmed,851,2036,106
...,...,...,...,...,...,...
35045517,7525,5807407,Confirmed,756,655,84
35045518,7525,5807407,Confirmed,767,779,84
35045519,7525,5807407,Confirmed,824,1043,4777
35045520,7525,5807407,Confirmed,815,924,2100


In [ ]:
csv_buffer = io.StringIO()
df_annotations.to_csv(csv_buffer, index=False)

s3_key = f"{prefix}/temporary/df_annotations.csv"

# Upload to S3
s3.put_object(
    Bucket=bucket_name, Key=s3_key, Body=csv_buffer.getvalue(), ContentType="text/csv"
)

{'ResponseMetadata': {'RequestId': '7801R8ZJHJEWF9DV',
  'HostId': '2LPWoeYPqpVJ89t524gIhmr3axzWijyUZdcHWvxENjs9BEmzw+IvYCV9BlVSNM/fiIcObc56SoU=',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'x-amz-id-2': '2LPWoeYPqpVJ89t524gIhmr3axzWijyUZdcHWvxENjs9BEmzw+IvYCV9BlVSNM/fiIcObc56SoU=',
   'x-amz-request-id': '7801R8ZJHJEWF9DV',
   'date': 'Sun, 03 May 2026 11:53:55 GMT',
   'x-amz-server-side-encryption': 'AES256',
   'etag': '"0fef7798e1e01155d2aec105d4ac3f20"',
   'x-amz-checksum-crc32': 'scX8Tw==',
   'x-amz-checksum-type': 'FULL_OBJECT',
   'content-length': '0',
   'server': 'AmazonS3'},
  'RetryAttempts': 0},
 'ETag': '"0fef7798e1e01155d2aec105d4ac3f20"',
 'ChecksumCRC32': 'scX8Tw==',
 'ChecksumType': 'FULL_OBJECT',
 'ServerSideEncryption': 'AES256'}

In [8]:
df_annotations["image_id"].nunique(), df_annotations["source_id"].nunique()

(1141551, 1311)

In [9]:
df_annotations["Status"].value_counts()

Status
Confirmed       21680834
Unconfirmed     13359126
Unclassified        5562
Name: count, dtype: int64

The numbers here are slightly lower than when only looking at the image lists, as this is an inner merge between the annotations and image lists, so images that have no annotations would be dropped.

In [9]:
df_annotations.groupby(["source_id", "image_id"], as_index=False).agg(
        status=("Status", "first"),
        status_nunique=("Status", "nunique"),
        annotation_count=("Status", "size"),
    )["status"].value_counts()

status
Confirmed       794477
Unconfirmed     346903
Unclassified       171
Name: count, dtype: int64

# Validate images
The image validation consists of 3 steps:
- Find all images that have an associated annotation
- Limit to only images which have confirmed annotations
- Make sure the image exists within the S3 Bucket 

In [10]:
df_images = df_annotations[df_annotations["Status"]=="Confirmed"].drop_duplicates(subset=["source_id", "image_id"]).reset_index()

In [ ]:
valid_list = []
for i, row in tqdm.tqdm(df_images.iterrows()):
    source_id = row["source_id"]
    image_id = row["image_id"]
    key = f"{prefix}/s{source_id}/images/{image_id}.jpg"

    response = s3.list_objects_v2(Bucket=bucket_name, Prefix=key, MaxKeys=1)

    if "Contents" in response:
        # print(f"Prefix exists: {prefix}")
        valid_list.append(True)
    else:
        # print(f"Prefix does not exist: {prefix}")
        valid_list.append(False)

df_images["validation"] = valid_list

In [11]:
df_images.shape

(794477, 7)

In [25]:
csv_buffer = io.StringIO()
df_images.to_csv(csv_buffer, index=False)

s3_key = f"{prefix}/temporary/df_images.csv"

# Upload to S3
s3.put_object(
    Bucket=bucket_name, Key=s3_key, Body=csv_buffer.getvalue(), ContentType="text/csv"
)

{'ResponseMetadata': {'RequestId': 'K74J8QD512JV4K5V',
  'HostId': '4tutLVoAns9c/MdYwOpf8YSE9AIeDw8yqGGr4Tkx6CA/ovP9/5+vhEqB3Yxd6TnCWChbrZJkrYP8yB1vkyJqHM+J3fYysNaaDUg9FNusLAU=',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'x-amz-id-2': '4tutLVoAns9c/MdYwOpf8YSE9AIeDw8yqGGr4Tkx6CA/ovP9/5+vhEqB3Yxd6TnCWChbrZJkrYP8yB1vkyJqHM+J3fYysNaaDUg9FNusLAU=',
   'x-amz-request-id': 'K74J8QD512JV4K5V',
   'date': 'Sun, 03 May 2026 12:23:50 GMT',
   'x-amz-server-side-encryption': 'AES256',
   'etag': '"b20ff1c5d456215123042c81581c09ff"',
   'x-amz-checksum-crc32': 'BBda2Q==',
   'x-amz-checksum-type': 'FULL_OBJECT',
   'content-length': '0',
   'server': 'AmazonS3'},
  'RetryAttempts': 0},
 'ETag': '"b20ff1c5d456215123042c81581c09ff"',
 'ChecksumCRC32': 'BBda2Q==',
 'ChecksumType': 'FULL_OBJECT',
 'ServerSideEncryption': 'AES256'}